In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(f"GROQ_API_KEY not found in {env_path} (cwd={Path.cwd()})")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7,
    max_tokens=800,
    max_retries=5,
    request_timeout=120,
)

In [4]:
from utils import get_serper_api_key

serper_api_key = get_serper_api_key()
if not serper_api_key:
    raise RuntimeError(f"SERPER_API_KEY not found in {env_path} (cwd={Path.cwd()})")

os.environ["SERPER_API_KEY"] = serper_api_key


In [5]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    llm=llm,
    verbose=True
)

In [6]:
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    llm=llm,
    verbose=True,
)

In [7]:
from crewai_tools import DirectoryReadTool, FileReadTool, SerperDevTool

In [8]:
directory_read_tool = DirectoryReadTool(directory="./instructions")
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [9]:
from crewai_tools import BaseTool

In [10]:
class SentimentAnalysisTool(BaseTool):
    name: str = ("Sentiment Analysis Tool",)
    description: str = (
        "Analyzes the sentiment of text to ensure positive and engaging communication."
    )

    def _run(self,text:str) -> str:
        # Your custom code tool goes here
        return "positive"

In [11]:
sentiment_analysis_tool = SentimentAnalysisTool()

In [12]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

In [13]:
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

In [14]:
crew = Crew(
    agents=[sales_rep_agent, lead_sales_rep_agent],
    tasks=[lead_profiling_task, personalized_outreach_task],
    verbose=2,
    # memory=True spins up crewai's RAG store, which embeds through OpenAI;
    # Groq has no embeddings endpoint, so leave it off.
    memory=False,
)

In [15]:
inputs = {
    "lead_name": "DeepLearning.AI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product lunch",
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Sales Representative
 [INFO]: == Starting Task: Conduct an in-depth analysis of DeepLearning.AI, a company in the Online Learning Platform sector that recently showed interest in our solutions. Utilize all available data sources to compile a detailed profile, focusing on key decision-makers, recent business developments, and potential needs that align with our offerings. This task is crucial for tailoring our engagement strategy effectively.
Don't make assumptions and only use information you absolutely sure about.


> Entering new CrewAgentExecutor chain...
Thought: I need to gather information about DeepLearning.AI to create a comprehensive profile. First, I should check if there are any existing internal files or instructions related to this lead or the general sales process. I will start by listing the files in the directory.
Action: List files in directory
Action Input: {} 

File paths: 
-./instructions/enterprise_solutions_framework.md
- ./instructions

2026-09-20 14:07:48,606 - 21808 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I have some general information, but I need more specific details about DeepLearning.AI's recent product launches, specific partnerships, and key personnel other than Andrew Ng. The previous search was a bit broad. Let me search for more specific information about their recent activities and leadership.
Action: Search the internet
Action Input: {"search_query": "DeepLearning.AI 2024 new courses AI Fund investment LandingAI partnership"} 


Search results: Title: DeepLearning.AI: Start or Advance Your Career in AI
Link: https://www.deeplearning.ai/
Snippet: AI Courses and Specializations. Build a foundation of machine learning and AI skills, and understand how to apply them in the real world.
---
Title: AI Fund raises $190M for new AI startups - LinkedIn
Link: https://www.linkedin.com/posts/andrewyng_im-delighted-to-announce-that-ai-fund-has-activity-7326246539029286912-ZCpd
Snippet: Venture funding in Q3 2025 is significantly influenced by artificial intelligence, with AI star

2026-09-20 14:08:23,796 - 21808 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I have gathered information about DeepLearning.AI's founding, scale (7 million learners), and their ecosystem (AI Fund, LandingAI, Coursera partnership). I also found a specific recent course "Document AI: From OCR to Agentic Doc Extraction" built with LandingAI, taught by David Park and Andrea Kropp. I need to dig deeper into their leadership structure beyond Andrew Ng and understand their specific business model and recent strategic moves to identify pain points or needs that CrewAI's solutions (likely AI automation/agent frameworks) could address. Let me search for their leadership team and recent strategic partnerships or product updates.
Action: Search the internet
Action Input: {"search_query": "DeepLearning.AI leadership team CEO COO CTO executive staff 2024"}

...

Thought: The previous search for leadership didn't yield specific results for DeepLearning.AI's internal executive team (CEO, CTO, etc.), mostly returning results for other companies or general AI leadership. I did notice a snippet from the Coursera blog mentioning Dr. Ng's role at Coursera ("Executive Chairman since August 2024"), which confirms his dual role. I need to find out who runs the day-to-day operations of DeepLearning.AI. Let me try a more specific search for "DeepLearning.AI CEO" or "DeepLearning.AI management team".
Action: Search the internet
Action Input: {"search_query": "who is the CEO of DeepLearning.AI management team"} 


Search results: Title: Andrew Ng - DeepLearning.AI, AI Fund and AI Aspire - LinkedIn
Link: https://www.linkedin.com/in/andrewyng
Snippet: I served as the company's Co-CEO in the early years with day-to-day responsibility for product, engineering, sales, marketing and operations, and remain ...
---
Title: About - DeepLearning.AI
Link: https://www.d

2026-09-20 14:10:19,389 - 21808 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: The search results confirm that Andrew Ng is the Founder and has held key leadership roles. One snippet from LinkedIn mentions he "served as the company's Co-CEO in the early years... and remain [involved]". This suggests he might not be the sole CEO now, or the title has evolved. The Coursera blog mentioned he is "Executive Chairman" there. I need to clarify his current specific title at DeepLearning.AI and find out who else is in key decision-making positions (like a VP of Product, Head of Partnerships, or CTO) to tailor our outreach. I also need to understand their specific recent strategic focus, particularly regarding AI agents and enterprise solutions, as that aligns with CrewAI's offerings. Let me search for "DeepLearning.AI AI Agent Short Courses" and "DeepLearning.AI enterprise solutions".

Action: Search the internet
Action Input: {"search_query": "DeepLearning.AI AI Agent Short Courses Enterprise Solutions 2024 2025"} 


Search results: Title: Agentic AI - DeepLearn

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01m2wtajwgem9r6g7bvbprbjjn` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 7144, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

2026-09-20 14:10:54,590 - 21808 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
